[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/05-random-forest-clasificacion.ipynb)


# Machine Learning: Random Forest para Clasificación

## Colab completo con datos reales, métricas, gráficos, optimización y predicciones

### Objetivo general

Aprender a construir, evaluar e interpretar un modelo de **Random Forest para Clasificación** utilizando exactamente el mismo dataset real de cáncer de mama empleado en el ejercicio de Árbol de Decisión.

### Al finalizar podrás:

- comprender Bagging, Bootstrap y aleatoriedad de variables;
- entrenar un `RandomForestClassifier`;
- evaluar Accuracy, Precision, Recall, Especificidad, F1 y ROC-AUC;
- interpretar matriz de confusión;
- analizar falsos positivos y falsos negativos;
- usar OOB Score;
- detectar underfitting y overfitting;
- estudiar `n_estimators` y `max_depth`;
- optimizar con `GridSearchCV`;
- realizar predicciones individuales y probabilidades;
- interpretar curvas ROC y Precision-Recall;
- analizar importancia de variables.

> Este notebook es académico y no debe utilizarse para decisiones clínicas reales.



# 1. ¿Qué es Random Forest para Clasificación?

Random Forest combina muchos Árboles de Decisión.

Cada árbol:

1. recibe una muestra bootstrap del conjunto Train;
2. utiliza subconjuntos aleatorios de variables;
3. genera una predicción de clase.

El bosque combina los votos de todos los árboles.

En clasificación también podemos obtener probabilidades con `predict_proba()`.



# 2. Árbol de Decisión vs Random Forest

| Aspecto | Árbol | Random Forest |
|---|---|---|
| Número de árboles | 1 | Muchos |
| Interpretabilidad | Alta | Menor |
| Estabilidad | Menor | Mayor |
| Varianza | Mayor | Menor |
| Overfitting | Más probable | Generalmente menor |
| Probabilidades | Sí | Sí |
| Feature importance | Sí | Sí |
| OOB Score | No | Sí |

Random Forest sacrifica parte de la interpretabilidad para ganar estabilidad.



# 3. Conceptos fundamentales

## Bagging
Entrenar varios modelos sobre muestras diferentes y combinar sus resultados.

## Bootstrap
Cada árbol aprende de una muestra aleatoria con reemplazo.

## Aleatoriedad de variables
En cada división se consideran solo algunas variables.

## Votación
Cada árbol vota por una clase.

## OOB — Out-of-Bag
Las observaciones que no entraron en la muestra bootstrap de un árbol pueden utilizarse como evaluación adicional.



# 4. Dataset real: Breast Cancer Wisconsin

Usaremos exactamente el mismo dataset del Árbol de Decisión para Clasificación:

- 569 observaciones
- 30 variables predictoras
- 2 clases:
  - malignant
  - benign

Así la comparación futura entre ambos modelos será directa.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    classification_report,
    precision_recall_curve,
    average_precision_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

data = load_breast_cancer(as_frame=True)

X = data.data.copy()
y = data.target.copy()

df = X.copy()
df["target"] = y

print("Dimensiones:", df.shape)
print(dict(enumerate(data.target_names)))
display(df.head())


# 5. Exploración inicial — EDA

In [ ]:

print("Valores faltantes:", df.isna().sum().sum())
display(df.describe().T.head(10))


In [ ]:

conteo = y.value_counts().sort_index()

tabla_clases = pd.DataFrame({
    "Clase": [data.target_names[i] for i in conteo.index],
    "Cantidad": conteo.values,
    "Porcentaje": conteo.values / len(y) * 100
})

display(tabla_clases.round(2))

plt.figure(figsize=(7,5))
plt.bar(tabla_clases["Clase"], tabla_clases["Cantidad"])
plt.title("Distribución de clases")
plt.ylabel("Cantidad")
plt.show()



# 6. Train/Test con stratify

Usaremos:

- 80% Train
- 20% Test
- `stratify=y`

Esto conserva aproximadamente la proporción de clases.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nDistribución Train")
print(y_train.value_counts(normalize=True).sort_index())

print("\nDistribución Test")
print(y_test.value_counts(normalize=True).sort_index())


# 7. Random Forest baseline

In [ ]:

modelo_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=2,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

modelo_baseline.fit(X_train, y_train)

pred_train_baseline = modelo_baseline.predict(X_train)
pred_test_baseline = modelo_baseline.predict(X_test)

proba_train_baseline = modelo_baseline.predict_proba(X_train)[:, 1]
proba_test_baseline = modelo_baseline.predict_proba(X_test)[:, 1]

print("OOB Score:", round(modelo_baseline.oob_score_, 4))



# 8. Matriz de confusión

- TP: verdadero positivo
- TN: verdadero negativo
- FP: falso positivo
- FN: falso negativo

La matriz permite entender qué tipo de errores comete el modelo.


In [ ]:

cm = confusion_matrix(y_test, pred_test_baseline)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Baseline")
plt.show()



# 9. Métricas

## Accuracy
Porcentaje total de aciertos.

## Precision
De los positivos predichos, cuántos eran realmente positivos.

## Recall / Sensibilidad
De los positivos reales, cuántos detectó.

## Especificidad
De los negativos reales, cuántos reconoció.

## F1
Equilibra Precision y Recall.

## ROC-AUC
Mide capacidad global de separación entre clases.


In [ ]:

def metricas_clasificacion(y_true, y_pred, y_proba=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    resultado = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_true, y_pred)
    }

    if y_proba is not None:
        resultado["ROC_AUC"] = roc_auc_score(y_true, y_proba)

    return resultado


In [ ]:

metricas_train_baseline = metricas_clasificacion(
    y_train,
    pred_train_baseline,
    proba_train_baseline
)

metricas_test_baseline = metricas_clasificacion(
    y_test,
    pred_test_baseline,
    proba_test_baseline
)

display(pd.DataFrame({
    "Train": metricas_train_baseline,
    "Test": metricas_test_baseline
}).T.round(4))


In [ ]:

print(classification_report(
    y_test,
    pred_test_baseline,
    target_names=data.target_names
))



# 10. ¿Qué métrica priorizar?

- FP costosos → **Precision**
- FN costosos → **Recall**
- equilibrio → **F1**
- evaluación general → **ROC-AUC**
- clases balanceadas → Accuracy puede ser útil, pero nunca sola.


# 11. Número de árboles vs desempeño

In [ ]:

cantidades_arboles = [10, 25, 50, 100, 200, 300]

accuracy_vals = []
f1_vals = []
auc_vals = []

for n in cantidades_arboles:
    modelo = RandomForestClassifier(
        n_estimators=n,
        max_depth=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    modelo.fit(X_train, y_train)

    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]

    accuracy_vals.append(accuracy_score(y_test, pred))
    f1_vals.append(f1_score(y_test, pred))
    auc_vals.append(roc_auc_score(y_test, proba))

plt.figure(figsize=(9,5))
plt.plot(cantidades_arboles, accuracy_vals, marker="o", label="Accuracy")
plt.plot(cantidades_arboles, f1_vals, marker="o", label="F1")
plt.plot(cantidades_arboles, auc_vals, marker="o", label="ROC-AUC")
plt.xlabel("n_estimators")
plt.ylabel("Score")
plt.title("Número de árboles vs desempeño")
plt.legend()
plt.grid(alpha=0.2)
plt.show()



Después de cierto número de árboles, el rendimiento suele estabilizarse.
Más árboles aumentan el costo computacional y no garantizan una mejora importante.


# 12. Profundidad vs overfitting

In [ ]:

profundidades = [2, 3, 4, 5, 6, 8, None]

acc_train = []
acc_test = []

for depth in profundidades:
    modelo = RandomForestClassifier(
        n_estimators=150,
        max_depth=depth,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    modelo.fit(X_train, y_train)

    acc_train.append(
        accuracy_score(y_train, modelo.predict(X_train))
    )
    acc_test.append(
        accuracy_score(y_test, modelo.predict(X_test))
    )

labels = [str(x) if x is not None else "None" for x in profundidades]
pos = np.arange(len(labels))

plt.figure(figsize=(9,5))
plt.plot(pos, acc_train, marker="o", label="Train")
plt.plot(pos, acc_test, marker="o", label="Test")
plt.xticks(pos, labels)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Profundidad vs Accuracy")
plt.legend()
plt.grid(alpha=0.2)
plt.show()



# 13. Hiperparámetros principales

- `n_estimators`: número de árboles.
- `max_depth`: profundidad máxima.
- `min_samples_split`: mínimo para dividir un nodo.
- `min_samples_leaf`: mínimo por hoja.
- `max_features`: variables consideradas en cada división.
- `criterion`: Gini o Entropía.
- `bootstrap`: activa bootstrap.
- `class_weight`: pondera clases.


# 14. Validación cruzada estratificada

In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc"
}

resultado_cv = cross_validate(
    modelo_baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

resumen_cv = pd.DataFrame({
    "Accuracy": resultado_cv["test_Accuracy"],
    "Precision": resultado_cv["test_Precision"],
    "Recall": resultado_cv["test_Recall"],
    "F1": resultado_cv["test_F1"],
    "ROC_AUC": resultado_cv["test_ROC_AUC"]
})

display(resumen_cv.round(4))
display(resumen_cv.mean().to_frame("Promedio").T.round(4))



# 15. Optimización con GridSearchCV

Mantendremos una cuadrícula razonable para que pueda ejecutarse cómodamente en Google Colab.

Usaremos **F1-score** como métrica principal.


In [ ]:

modelo_grid = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, None],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", 0.7],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    estimator=modelo_grid,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Mejores hiperparámetros:")
print(grid.best_params_)

print("\nMejor F1 promedio de CV:")
print(round(grid.best_score_, 4))


# 16. Evaluación final

In [ ]:

mejor_modelo = grid.best_estimator_

pred_train_final = mejor_modelo.predict(X_train)
pred_test_final = mejor_modelo.predict(X_test)

proba_train_final = mejor_modelo.predict_proba(X_train)[:, 1]
proba_test_final = mejor_modelo.predict_proba(X_test)[:, 1]

metricas_train_final = metricas_clasificacion(
    y_train,
    pred_train_final,
    proba_train_final
)

metricas_test_final = metricas_clasificacion(
    y_test,
    pred_test_final,
    proba_test_final
)

display(pd.DataFrame({
    "Train": metricas_train_final,
    "Test": metricas_test_final
}).T.round(4))


In [ ]:

display(pd.DataFrame({
    "Baseline": metricas_test_baseline,
    "Optimizado": metricas_test_final
}).T.round(4))


# 17. Matriz de confusión final

In [ ]:

cm_final = confusion_matrix(y_test, pred_test_final)

ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Random Forest optimizado")
plt.show()

tn, fp, fn, tp = cm_final.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


# 18. Curva ROC

In [ ]:

fpr, tpr, thresholds = roc_curve(
    y_test,
    proba_test_final
)

auc = roc_auc_score(
    y_test,
    proba_test_final
)

plt.figure(figsize=(7,6))
plt.plot(
    fpr,
    tpr,
    label=f"Random Forest (AUC = {auc:.3f})"
)
plt.plot([0,1], [0,1], linestyle="--", label="Azar")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 19. Curva Precision-Recall

In [ ]:

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(
    y_test,
    proba_test_final
)

ap = average_precision_score(
    y_test,
    proba_test_final
)

plt.figure(figsize=(7,6))
plt.plot(
    recall_curve,
    precision_curve,
    label=f"Average Precision = {ap:.3f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 20. Predicción individual

In [ ]:

posicion = 0

observacion = X_test.iloc[[posicion]]
valor_real = y_test.iloc[posicion]

clase_predicha = mejor_modelo.predict(observacion)[0]
probabilidades = mejor_modelo.predict_proba(observacion)[0]

print("CLASE REAL:", data.target_names[valor_real])
print("CLASE PREDICHA:", data.target_names[clase_predicha])

print("\nPROBABILIDADES:")
for clase, prob in zip(data.target_names, probabilidades):
    print(f"{clase}: {prob:.2%}")

display(observacion)


# 21. Predicciones para varias observaciones

In [ ]:

n_ejemplos = min(15, len(X_test))

muestra_test = X_test.sample(
    n=n_ejemplos,
    random_state=RANDOM_STATE
)

real_muestra = y_test.loc[muestra_test.index]
pred_muestra = mejor_modelo.predict(muestra_test)
proba_muestra = mejor_modelo.predict_proba(muestra_test)[:, 1]

tabla_predicciones = pd.DataFrame({
    "Real": [data.target_names[i] for i in real_muestra.to_numpy()],
    "Predicción": [data.target_names[i] for i in pred_muestra],
    "Probabilidad_clase_1": proba_muestra
}, index=muestra_test.index)

tabla_predicciones["Correcta"] = (
    tabla_predicciones["Real"]
    == tabla_predicciones["Predicción"]
)

display(tabla_predicciones)


# 22. Correctas vs incorrectas

In [ ]:

aciertos = (
    pred_test_final
    == y_test.to_numpy()
)

tabla_aciertos = pd.DataFrame({
    "Resultado": ["Correctas", "Incorrectas"],
    "Cantidad": [
        aciertos.sum(),
        (~aciertos).sum()
    ]
})

display(tabla_aciertos)

plt.figure(figsize=(7,5))
plt.bar(
    tabla_aciertos["Resultado"],
    tabla_aciertos["Cantidad"]
)
plt.title("Predicciones correctas vs incorrectas")
plt.ylabel("Observaciones")
plt.show()


# 23. Importancia de variables

In [ ]:

importancias = pd.Series(
    mejor_modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(
    importancias
    .head(15)
    .to_frame("Importancia")
)


In [ ]:

top_importancias = importancias.head(15)

plt.figure(figsize=(10,6))
plt.barh(
    top_importancias.index[::-1],
    top_importancias.values[::-1]
)
plt.title("15 variables más importantes")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()



> La importancia de variables es predictiva, no causal.


# 24. OOB Score del mejor modelo

In [ ]:

params_finales = grid.best_params_.copy()

modelo_oob = RandomForestClassifier(
    **params_finales,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

modelo_oob.fit(X_train, y_train)

print(
    "OOB Score del modelo final:",
    round(modelo_oob.oob_score_, 4)
)



# 25. Random Forest en la vida real

## Fraude
fraude / no fraude

## Crédito
aprobado / rechazado

## Marketing
compra / no compra

## Industria
falla / no falla

## Salud
clase diagnóstica

Random Forest es especialmente útil en datos tabulares con relaciones no lineales.



# 26. ¿Cuándo elegir Random Forest?

Puede ser una muy buena opción cuando:

- un árbol es demasiado inestable;
- hay interacciones;
- existen relaciones no lineales;
- buscamos un baseline fuerte;
- priorizamos rendimiento sobre interpretabilidad total.

Un único árbol puede seguir siendo preferible si la explicación de reglas es crítica.



# 27. Métricas recomendadas

- falsos negativos costosos → Recall
- falsos positivos costosos → Precision
- equilibrio → F1
- evaluación global → ROC-AUC
- clases balanceadas → Accuracy como complemento
- evaluación adicional → OOB Score



# 28. Errores comunes

1. Usar solo Accuracy.
2. Ignorar FP y FN.
3. No utilizar stratify.
4. Pensar que más árboles siempre mejora mucho.
5. Evaluar únicamente Train.
6. Optimizar mirando Test.
7. Confundir probabilidad con certeza.
8. Interpretar importancia como causalidad.
9. Creer que Random Forest nunca sobreajusta.
10. Llevar un modelo académico directamente a decisiones clínicas.



# 29. Flujo recomendado

**Problema → clases → EDA → balance → Train/Test estratificado → baseline → métricas → matriz de confusión → OOB → validación cruzada → GridSearchCV → Test final → ROC → Precision-Recall → predicciones → importancia**



# 30. Conclusión

Random Forest para clasificación:

- combina muchos árboles;
- usa bootstrap;
- utiliza aleatoriedad de variables;
- genera clases y probabilidades;
- suele ser más estable que un árbol individual;
- permite OOB Score;
- debe evaluarse con varias métricas;
- requiere analizar FP y FN;
- puede optimizarse con validación cruzada.

La idea principal es:

> **Un buen clasificador no es simplemente el que tiene más Accuracy, sino el que comete los errores más aceptables para el problema real.**



# 31. Referencias

- Scikit-learn — RandomForestClassifier  
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

- Scikit-learn — Breast Cancer Wisconsin dataset  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

- Scikit-learn — Classification metrics  
  https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics

- Scikit-learn — Ensemble methods  
  https://scikit-learn.org/stable/modules/ensemble.html
